In [2]:
import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix, lil_matrix
from pyproj import Transformer
from sklearn.preprocessing import StandardScaler
import pyreadr 
from pathlib import Path
import os

os.chdir(Path.cwd().parent)

In [ ]:
# ================================================================
# Imports
# ================================================================
import numpy as np
from tqdm import tqdm
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix, coo_matrix, diags
from sksparse.cholmod import cholesky
from polyagamma import random_polyagamma
import geopandas as gpd
import pyreadr

# ================================================================
# Load data (remove isolated points)
# ================================================================
no_nbs = np.array([
    57,170,236,269,343,685,946,947,989,
    1037,1084,1090,1109,1118,1127,1176,1203
]) - 1

snow_cleaned_full = pyreadr.read_r("snow_cleaned_full.Rda")
snow_cleaned_full = list(snow_cleaned_full.values())[0]

all_y = snow_cleaned_full.drop(index=no_nbs).reset_index(drop=True)

coords = all_y.iloc[:, :2].to_numpy()
y      = all_y.iloc[:, 2:].to_numpy()

S, TT = y.shape
period = 52

# ================================================================
# Global time trend (scaled once)
# ================================================================
t_full = np.arange(1, TT + 1)
t_trend_full = (t_full - t_full.mean()) / t_full.std(ddof=0)

# ================================================================
# Build adjacency + ICAR precision
# ================================================================
gdf = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(coords[:, 0], coords[:, 1]),
    crs="EPSG:4326"
)
gdf_aeqd = gdf.to_crs("+proj=aeqd +lat_0=90 +lon_0=-100")
xy = np.vstack([gdf_aeqd.geometry.x, gdf_aeqd.geometry.y]).T / 1e6

Distances = squareform(pdist(xy))
A = (Distances <= 0.22).astype(int)
np.fill_diagonal(A, 0)
A = csr_matrix(A)

deg = np.array(A.sum(axis=1)).flatten()
Q_icar = diags(deg) - A
I_S    = diags(np.ones(S))

# ================================================================
# Event: p01
# ================================================================
loc_mask = (y[:, :-1] == 0)
loc = np.where(loc_mask)

row_idx  = loc[0]
time_idx = loc[1]                 # ∈ {0,...,TT-2}
N = len(row_idx)

next_y = y[row_idx, time_idx + 1]
kappa  = next_y - 0.5

# ================================================================
# Covariates
# ================================================================
t_raw   = time_idx + 1
t_trend = t_trend_full[time_idx]

cov = np.column_stack([
    np.ones(N), np.ones(N),
    np.cos(2*np.pi*t_raw / period),
    np.cos(2*np.pi*t_raw / period),
    np.sin(2*np.pi*t_raw / period),
    np.sin(2*np.pi*t_raw / period),
    t_trend, t_trend
])
K = cov.shape[1]

# ================================================================
# Design matrix X0
# ================================================================
rows, cols, vals = [], [], []
for i in tqdm(range(N), desc="Building X0"):
    s = row_idx[i]
    for k in range(K):
        rows.append(i)
        cols.append(s + k*S)
        vals.append(cov[i, k])

X0 = coo_matrix((vals, (rows, cols)), shape=(N, K*S)).tocsr()

# ================================================================
# Prior blocks for theta
# ================================================================
prior_blocks = [Q_icar if k % 2 == 0 else I_S for k in range(K)]

# ================================================================
# Week index for observations
# ================================================================
week_obs = time_idx % period   # length N

# ================================================================
# MCMC settings
# ================================================================
burn = 1000
thin = 5
tot_save = 1000
total_iters = burn + tot_save * thin

# ================================================================
# Initialization (BYM with weekly tau)
# ================================================================
theta_u = np.random.randn(K, S)   # structured
theta_v = np.random.randn(K, S)   # iid

tau1 = np.ones((K, period))       # ICAR precision
tau2 = np.ones((K, period))       # iid precision

# IG hyperparameters
a1, b1 = 2.0, 1.0
a2, b2 = 2.0, 1.0

all_theta_u = np.zeros((K, S, tot_save))
all_theta_v = np.zeros((K, S, tot_save))
all_tau1    = np.zeros((K, period, tot_save))
all_tau2    = np.zeros((K, period, tot_save))

save_idx = 0
ridge_eps = 1e-6
ridge_hits = 0

# ================================================================
# MCMC
# ================================================================
for it in tqdm(range(total_iters), desc="MCMC p01 (weekly tau BYM)"):

    # ------------------------------------------------------------
    # Linear predictor
    # ------------------------------------------------------------
    phi = np.zeros(N)
    for k in range(K):
        Xk = X0[:, k*S:(k+1)*S]
        phi += Xk @ (theta_u[k] + theta_v[k])

    # ------------------------------------------------------------
    # Polya–Gamma
    # ------------------------------------------------------------
    omega = random_polyagamma(1, phi)

    # ------------------------------------------------------------
    # Update theta_u, theta_v (BYM)
    # ------------------------------------------------------------
    for k in range(K):

        Xk = X0[:, k*S:(k+1)*S]
        wk = omega
        kk = kappa

        # contribution of other components
        phi_minus = phi - Xk @ (theta_u[k] + theta_v[k])

        # working response
        z = kk / wk + phi

        # ---------- structured ----------
        w_eff = wk
        XtW = Xk.T.multiply(w_eff)
        Prec_u = XtW @ Xk + tau1[k, week_obs[0]] * Q_icar

        try:
            factor_u = cholesky(Prec_u)
        except:
            ridge_hits += 1
            factor_u = cholesky(Prec_u + ridge_eps * I_S)

        mu_u = factor_u.solve_A(Xk.T @ (w_eff * (z - phi_minus)))
        theta_u[k] = mu_u + factor_u.solve_A(np.random.randn(S))

        # ---------- iid ----------
        Prec_v = XtW @ Xk + tau2[k, week_obs[0]] * I_S
        try:
            factor_v = cholesky(Prec_v)
        except:
            ridge_hits += 1
            factor_v = cholesky(Prec_v + ridge_eps * I_S)

        mu_v = factor_v.solve_A(Xk.T @ (w_eff * (z - phi_minus)))
        theta_v[k] = mu_v + factor_v.solve_A(np.random.randn(S))

    # ------------------------------------------------------------
    # Update tau1, tau2 (weekly, covariate-specific)
    # ------------------------------------------------------------
    for k in range(K):
        for w in range(period):

            # structured
            quad_u = theta_u[k] @ (Q_icar @ theta_u[k])
            tau1[k, w] = np.random.gamma(
                a1 + 0.5 * (S - 1),
                1.0 / (b1 + 0.5 * quad_u)
            )

            # iid
            quad_v = np.sum(theta_v[k]**2)
            tau2[k, w] = np.random.gamma(
                a2 + 0.5 * S,
                1.0 / (b2 + 0.5 * quad_v)
            )

    # ------------------------------------------------------------
    # Save
    # ------------------------------------------------------------
    if it >= burn and (it - burn) % thin == 0:
        all_theta_u[:, :, save_idx] = theta_u
        all_theta_v[:, :, save_idx] = theta_v
        all_tau1[:, :, save_idx]    = tau1
        all_tau2[:, :, save_idx]    = tau2
        save_idx += 1
        if save_idx == tot_save:
            break

# ================================================================
# Save results
# ================================================================
np.savez_compressed(
    "bym01_weeklyTau_fullscript.npz",
    theta_u=all_theta_u,
    theta_v=all_theta_v,
    tau1=all_tau1,
    tau2=all_tau2,
    ridge_hits=ridge_hits
)

print("DONE. Ridge used:", ridge_hits)



MCMC p01 (weekly a):   0%|          | 0/6000 [00:00<?, ?it/s]C:\Users\lix23\AppData\Local\Temp\ipykernel_29376\3208014720.py:168: CholmodTypeConversionWarning: converting matrix of class csr_matrix to CSC format
  factor = cholesky(post_prec)
MCMC p01 (weekly a): 100%|█████████▉| 5995/6000 [4:18:22<00:12,  2.59s/it]  


DONE. Ridge used (theta): 0
